In [23]:
import torch
import torch.nn.functional as F
import torch.nn as nn

In [ ]:
# EX1: The n-dimensional tensor mastery challenge: 
# Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel, 
# treating the heads as another batch dimension (answer is in nanoGPT).

In [ ]:
# 平均聚合
torch.manual_seed(12)
X = torch.randint(0,5, (4,3)).float()

avgS = torch.tril(torch.ones(4,4))
tril = torch.tril(torch.ones(4,4))
wei = avgS.masked_fill(tril == 0, float('-inf'))
avgA = F.softmax(wei, dim=1)

Y = avgA @ X
print(f"X->{X} \n A->{avgA} \n Y->{Y}")

X->tensor([[3., 3., 2.],
        [2., 3., 1.],
        [4., 3., 3.],
        [0., 4., 3.]]) 
 A->tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]]) 
 Y->tensor([[3.0000, 3.0000, 2.0000],
        [2.5000, 3.0000, 1.5000],
        [3.0000, 3.0000, 2.0000],
        [2.2500, 3.2500, 2.2500]])


In [ ]:
# 增加 Q，K, V
QW = torch.randn(3,2)
KW = torch.randn(3,2)

Q = X @ QW
K = X @ KW

S = Q @ K.T # 替代之前的都是1的S

tril = torch.tril(torch.ones(4,4))
wei = S.masked_fill(tril == 0, float('-inf'))
A = F.softmax(wei, dim=1)

VW = torch.randn(3,3)
V = X @ VW
Y = A @ V

S, A, Y

(tensor([[ 13.3999,  11.3605,  15.4393,  13.0266],
         [ 10.6214,   8.8882,  12.3546,   9.5303],
         [ 16.1784,  13.8328,  18.5240,  16.5229],
         [-10.4806,  -8.2867, -12.6746,  -6.1076]]),
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
         [0.8498, 0.1502, 0.0000, 0.0000],
         [0.0867, 0.0083, 0.9050, 0.0000],
         [0.0112, 0.1004, 0.0012, 0.8872]]),
 tensor([[-3.1957,  1.6778,  3.0454],
         [-3.2115,  1.6837,  2.7983],
         [-3.1012,  1.6426,  4.5207],
         [-5.5198,  0.5012,  2.0031]]))

In [6]:
# 缩放
scaleS = S/(2**0.5)
scaleWei = scaleS.masked_fill(tril == 0, float('-inf'))
scaleA = F.softmax(scaleWei, dim=1)
scaleA

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.7730, 0.2270, 0.0000, 0.0000],
        [0.1552, 0.0296, 0.8152, 0.0000],
        [0.0358, 0.1688, 0.0076, 0.7879]])

In [7]:
d = 2 # head size
Xin = torch.rand(2,4,3)
QW = torch.rand(3,d)
KW = torch.rand(3,d)
VW = torch.rand(3,d)

Q = Xin @ QW
K = Xin @ KW
V = Xin @ VW

wei = (Q @ K.transpose(-2, -1)) / (d**0.5)
maskWei = wei.masked_fill(tril == 0, float('-inf'))
A = F.softmax(maskWei, dim=-1)
Y = A @ V

In [8]:
Xin.shape, Q.shape, K.shape, V.shape, A.shape, Y.shape

(torch.Size([2, 4, 3]),
 torch.Size([2, 4, 2]),
 torch.Size([2, 4, 2]),
 torch.Size([2, 4, 2]),
 torch.Size([2, 4, 4]),
 torch.Size([2, 4, 2]))

In [ ]:
# 单头
class SingleHeadAttention(nn.Module):

    def __init__(self, head_size, embed_dim):
        super().__init__()
        self.head_size = head_size
        self.query = nn.Linear(embed_dim, head_size, False)
        self.key = nn.Linear(embed_dim, head_size, False)
        self.value = nn.Linear(embed_dim, head_size, False)


    def forward(self, x):
        B, T, C = x.shape
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        mask = torch.tril(torch.ones(T,T))
        score = Q @ K.transpose(-2, -1)
        score = score * (self.head_size ** -0.5)
        score = score.masked_fill(mask == 0, float('-inf'))
        atten_weights = F.softmax(score, dim=-1)
        return atten_weights @ V


In [50]:
class MultiHead_v1(nn.Module):
    def __init__(self, num_head, head_size, emb_dim):
        super().__init__()
        self.ml = nn.ModuleList([SingleHeadAttention(head_size, emb_dim) for _ in range(num_head)])
        self.linear = nn.Linear(num_head * head_size, emb_dim, bias = False)

    def forward(self, x):
        out_list = []
        for h in self.ml:
            out_list.append(h(x))
        cat_out = torch.cat(out_list, dim=-1)
        out = self.linear(cat_out)
        return out

In [ ]:
# 单头整合到多头，nanoGPT里还进行了进一步的整合
class MultiHead_v2(nn.Module):
    def __init__(self, num_head, head_dim, emb_dim):
        super().__init__()
        self.num_head = num_head
        self.head_dim = head_dim
        self.Q_proj = nn.Linear(emb_dim, num_head * head_dim, bias=False)
        self.K_proj = nn.Linear(emb_dim, num_head * head_dim, bias=False)
        self.V_proj = nn.Linear(emb_dim, num_head * head_dim, bias=False)
        self.linear = nn.Linear(head_dim * num_head, emb_dim, bias=False)

    # 维度拆分，(B,T,C) => (B,H,T,D)
    def tran(self, batch_size, time_size, input : torch.Tensor):
        out = input.view(batch_size, time_size, self.num_head, self.head_dim) # (B,T,H,D)
        out = out.transpose(1, 2)
        return out

    def forward(self, x : torch.Tensor):
        B, T, C = x.shape

        Q = self.tran(B,T,self.Q_proj(x))
        K = self.tran(B,T,self.K_proj(x))
        V = self.tran(B,T,self.V_proj(x))

        S = Q @ K.transpose(-1,-2) * (self.head_dim **(-0.5))
        mask = torch.tril(torch.ones(T, T))
        S = S.masked_fill(mask==0, float('-inf'))
        A = F.softmax(S, dim=-1)
        O = A @ V

        O = O.transpose(1,2) # 先交换维度
        O = O.reshape(B, T, self.num_head * self.head_dim)
        out = self.linear(O)
        return out


In [ ]:
# 测试输入
X = torch.randint(0,5, (2,4,6)).float()
X

tensor([[[4., 1., 2., 1., 1., 2.],
         [3., 2., 1., 4., 0., 1.],
         [1., 1., 1., 4., 4., 1.],
         [4., 2., 3., 3., 2., 0.]],

        [[3., 4., 2., 1., 0., 2.],
         [1., 1., 0., 0., 4., 3.],
         [4., 4., 2., 1., 2., 2.],
         [4., 0., 3., 3., 1., 4.]]])

In [ ]:
# 用相同版本验证结果相同
mul_1 = MultiHead_v1(3, 2, 6)
mul_2 = MultiHead_v2(3, 2, 6)

q_weights = []
k_weights = []
v_weights = []

for head in mul_1.ml:
    q_weights.append(head.query.weight)
    k_weights.append(head.key.weight)
    v_weights.append(head.value.weight)

QW_cat = torch.cat(q_weights, dim=0)
KW_cat = torch.cat(k_weights, dim=0)
VW_cat = torch.cat(v_weights, dim=0)
LW = mul_1.linear.weight

with torch.no_grad():
    mul_2.Q_proj.weight.copy_(QW_cat)
    mul_2.K_proj.weight.copy_(KW_cat)
    mul_2.V_proj.weight.copy_(VW_cat)
    mul_2.linear.weight.copy_(LW)

out1 = mul_1(X)
out2 = mul_2(X)

out1, out2


(tensor([[[ 0.4935, -0.4151, -1.0253,  0.4438, -0.3064,  0.1227],
          [ 0.4935, -0.3625, -1.7065,  0.4876, -0.0142,  0.0498],
          [-0.3676,  0.3055, -1.0440,  0.3639,  0.0257,  0.0222],
          [-0.2889,  0.2057, -1.1779,  0.3994,  0.0228,  0.0397]],
 
         [[ 0.3676,  0.2139, -1.6141, -0.5353, -1.3061,  0.0125],
          [ 0.3895,  0.2764, -1.5350, -0.2469, -1.1408, -0.0041],
          [ 0.5443,  0.1562, -1.7844, -0.2161, -1.3039,  0.0260],
          [ 0.4918,  0.2365, -1.9282, -0.1751, -1.2222,  0.0045]]],
        grad_fn=<UnsafeViewBackward0>),
 tensor([[[ 0.4935, -0.4151, -1.0253,  0.4438, -0.3064,  0.1227],
          [ 0.4935, -0.3625, -1.7065,  0.4876, -0.0142,  0.0498],
          [-0.3676,  0.3055, -1.0440,  0.3639,  0.0257,  0.0222],
          [-0.2889,  0.2057, -1.1779,  0.3994,  0.0228,  0.0397]],
 
         [[ 0.3676,  0.2139, -1.6141, -0.5353, -1.3061,  0.0125],
          [ 0.3895,  0.2764, -1.5350, -0.2469, -1.1408, -0.0041],
          [ 0.5443,  0.1562,